# 트랜스포머와 어텐션 실습

**Transformer · Attention · 자기어텐션**

입력 요소들이 서로를 얼마나 참고할지 학습하는 어텐션 구조 기반 모델.

소재 분야에서 이해하기: 조성 수열이나 논문 문장을 처리하는 모델의 기본 구조로 쓰인다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [트랜스포머 원논문](https://arxiv.org/abs/1706.03762)

## 1. 어텐션 가중치를 직접 계산

토큰들이 서로를 얼마나 참고하는지 스케일드 닷프로덕트 어텐션으로 계산합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

tokens = ['Li', 'Ni', 'Mn', 'Co', 'O2']
dimension = 6
embedding = rng.normal(0, 1, (len(tokens), dimension))

def attention(embedding, seed=1):
    local = np.random.default_rng(seed)
    d = embedding.shape[1]
    Wq, Wk, Wv = (local.normal(0, 0.5, (d, d)) for _ in range(3))
    q, k, v = embedding @ Wq, embedding @ Wk, embedding @ Wv
    scores = q @ k.T / np.sqrt(d)
    weights = np.exp(scores - scores.max(1, keepdims=True))
    weights = weights / weights.sum(1, keepdims=True)
    return weights, weights @ v

weights, output = attention(embedding)
print('행별 합이 1인지 확인:', np.round(weights.sum(1), 3))

In [ ]:
plt.imshow(weights, cmap='viridis')
plt.colorbar(label='attention weight')
plt.xticks(range(len(tokens)), tokens); plt.yticks(range(len(tokens)), tokens)
plt.xlabel('attended to'); plt.ylabel('query'); plt.title('attention matrix'); plt.show()
for index, name in enumerate(tokens):
    top = int(np.argmax(weights[index]))
    print('%-3s 토큰이 가장 많이 참고한 토큰: %s (%.2f)' % (name, tokens[top], weights[index, top]))

## 2. 위치 정보 없이는 순서를 모릅니다

In [ ]:
shuffled = embedding[[2, 0, 4, 1, 3]]
weights_shuffled, output_shuffled = attention(shuffled)
print('원래 순서 출력 합 %.4f' % output.sum())
print('순서를 바꾼 출력 합 %.4f  (같은 집합이면 합은 보존됩니다)' % output_shuffled.sum())

position = np.arange(len(tokens))[:, None]
angles = position / np.power(10000, np.arange(dimension)[None, :] / dimension)
encoding = np.where(np.arange(dimension) % 2 == 0, np.sin(angles), np.cos(angles))
plt.imshow(encoding, cmap='coolwarm'); plt.colorbar(label='value')
plt.xlabel('embedding dimension'); plt.ylabel('position'); plt.title('positional encoding'); plt.show()

## 3. 해석

어텐션 자체는 순서를 모르기 때문에 위치 인코딩을 더해줍니다. 소재 분야에서는 조성 수열,
합성 절차 문장, 스펙트럼 시퀀스 등에 이 구조가 쓰입니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#transformer)을 여세요.